# Starbucks Rewards Data Cleaning

This notebook prepares the raw Starbucks Rewards files for downstream analysis. Each major cleaning operation is intentionally kept in its own code cell so that the effect of the operation can be inspected and reproduced.

The workflow is: load the three raw JSONL files, clean missing/null values, treat numeric outliers, remove informational coupons, and save the cleaned data. Informational offers are excluded because they do not contain a reward or a purchase requirement and are not the incentive treatments used by the later uplift analysis.

In [1]:
# Import only the libraries needed for paths, JSONL parsing, tabular manipulation, and reporting.
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Resolve the project root whether Jupyter starts in the repository root or in notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = RAW_DIR / 'profile.json'
PORTFOLIO_PATH = RAW_DIR / 'portfolio.json'
TRANSCRIPT_PATH = RAW_DIR / 'transcript.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed output directory: {PROCESSED_DIR}')

Project root: D:\SanJose\sem2\CMPE252
Processed output directory: D:\SanJose\sem2\CMPE252\data\processed


In [2]:
# Load each JSON Lines file into a DataFrame.
# A JSON Lines reader is used because every line in the raw files is one independent record.
def read_jsonl(path):
    with path.open(encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

profile = pd.DataFrame(read_jsonl(PROFILE_PATH))
portfolio = pd.DataFrame(read_jsonl(PORTFOLIO_PATH))
transcript = pd.DataFrame(read_jsonl(TRANSCRIPT_PATH))

# Expand the nested transcript value object into ordinary columns.
# Missing keys become NaN; this is expected because different event types have different payloads.
transcript_values = pd.json_normalize(transcript['value'].where(transcript['value'].notna(), None))
base_events = transcript.drop(columns='value').reset_index(drop=True)
transcript_values = transcript_values.reset_index(drop=True)
# The nested payload itself also has two historical spellings. Coalesce them before renaming.
if 'offer id' in transcript_values.columns and 'offer_id' in transcript_values.columns:
    transcript_values['offer_id'] = transcript_values['offer_id'].fillna(transcript_values['offer id'])
    transcript_values = transcript_values.drop(columns='offer id')
elif 'offer id' in transcript_values.columns:
    transcript_values = transcript_values.rename(columns={'offer id': 'offer_id'})
# The raw data uses top-level offer_id for completed events and nested offer id for
# received/viewed events. Coalesce both sources before concatenating to avoid duplicate columns.
if 'offer_id' in base_events.columns and 'offer_id' in transcript_values.columns:
    base_events['offer_id'] = base_events['offer_id'].fillna(transcript_values['offer_id'])
    transcript_values = transcript_values.drop(columns='offer_id')
events = pd.concat([base_events, transcript_values], axis=1)

# Attach offer_type to every event so that coupon-level filtering is possible later.
events = events.merge(portfolio[['id', 'offer_type']], left_on='offer_id', right_on='id', how='left')
events = events.drop(columns='id')

print(f'Loaded profiles: {len(profile):,}')
print(f'Loaded offers: {len(portfolio):,}')
print(f'Loaded events: {len(events):,}')
print('Event columns:', events.columns.tolist())

Loaded profiles: 17,000
Loaded offers: 10
Loaded events: 306,534
Event columns: ['person', 'event', 'time', 'amount', 'offer_id', 'reward', 'offer_type']


## 1. Clean missing and null values

In [3]:
# Treat the dataset-specific age value 118 as missing before calculating statistics.
# In the source data, age 118 is a sentinel for customers whose demographics were not supplied.
profile['age'] = profile['age'].replace(118, np.nan)

# Convert the membership date to a real datetime value. Invalid dates become NaT rather than
# silently remaining as strings, which makes later date calculations safer.
profile['became_member_on'] = pd.to_datetime(
    profile['became_member_on'].astype('string'), format='%Y%m%d', errors='coerce'
)

# Fill numeric demographic nulls with the median of the observed population. The median is
# robust to skew and avoids losing otherwise useful customer records.
profile['age'] = profile['age'].fillna(profile['age'].median())
profile['income'] = profile['income'].fillna(profile['income'].median())

# Replace a missing gender with an explicit category instead of guessing a gender.
profile['gender'] = profile['gender'].fillna('Unknown')

# Keep the event-specific nulls meaningful: transaction amounts and rewards are only defined
# for their corresponding event types, so fill absent numeric payloads with zero and absent
# offer identifiers with an explicit label.
events['amount'] = events['amount'].fillna(0.0)
events['reward'] = events['reward'].fillna(0.0)
events['offer_id'] = events['offer_id'].fillna('no_offer')
events['offer_type'] = events['offer_type'].fillna('transaction_or_unknown')

# Remove exact duplicate records only. This does not deduplicate legitimate repeated customer
# events, which can have identical values but represent separate observations.
profile = profile.drop_duplicates().reset_index(drop=True)
portfolio = portfolio.drop_duplicates(subset='id').reset_index(drop=True)
events = events.drop_duplicates().reset_index(drop=True)

print('Remaining null counts after treatment:')
print('profile:', profile.isna().sum().to_dict())
print('events:', events.isna().sum().to_dict())

Remaining null counts after treatment:
profile: {'gender': 0, 'age': 0, 'id': 0, 'became_member_on': 0, 'income': 0}
events: {'person': 0, 'event': 0, 'time': 0, 'amount': 0, 'offer_id': 0, 'reward': 0, 'offer_type': 0}


## 2. Numeric statistics before outlier treatment

In [4]:
# Calculate baseline statistics before any outlier treatment is applied.
# These values provide a direct comparison against the distributions after IQR clipping.
# Range is reported as minimum to maximum. Mean and median are calculated for age, income,
# and transaction amount so the effect of extreme values can be assessed clearly.
transaction_amount = events.loc[events['event'].eq('transaction'), 'amount']
before_outlier_stats = pd.DataFrame({
    'variable': ['profile.age', 'profile.income', 'transaction.amount'],
    'range_min': [profile['age'].min(), profile['income'].min(), transaction_amount.min()],
    'range_max': [profile['age'].max(), profile['income'].max(), transaction_amount.max()],
    'mean': [profile['age'].mean(), profile['income'].mean(), transaction_amount.mean()],
    'median': [profile['age'].median(), profile['income'].median(), transaction_amount.median()]
})

# Round only the displayed copy; the original statistics retain full precision.
display(before_outlier_stats.round(2))
before_outlier_stats.to_csv(PROCESSED_DIR / 'statistics_before_outliers.csv', index=False)

,variable,range_min,range_max,mean,median
0,profile.age,18.00,101.00,54.47,55.00
1,profile.income,30000.00,120000.00,65225.24,64000.00
2,transaction.amount,0.05,1062.28,12.78,8.89


## 3. Treat numeric outliers

In [5]:
# Use IQR winsorization: values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR] are clipped to the
# nearest boundary rather than deleting their customer/event rows. This preserves sample size
# while preventing extreme numeric values from dominating summary statistics and models.
def clip_iqr(series, name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    before = ((series < lower) | (series > upper)).sum()
    clipped = series.clip(lower=lower, upper=upper)
    print(f'{name}: bounds=({lower:.2f}, {upper:.2f}); values treated={before:,}')
    return clipped

# Age and income are profile-level numeric attributes. Transaction amount is clipped only
# for transaction events; non-transaction event amounts are zero by construction.
profile['age'] = clip_iqr(profile['age'], 'profile.age')
profile['income'] = clip_iqr(profile['income'], 'profile.income')
transaction_mask = events['event'].eq('transaction')
events.loc[transaction_mask, 'amount'] = clip_iqr(
    events.loc[transaction_mask, 'amount'], 'transaction.amount'
)

print('Outlier treatment completed without deleting rows.')

profile.age: bounds=(15.00, 95.00); values treated=48
profile.income: bounds=(13500.00, 113500.00); values treated=306
transaction.amount: bounds=(-20.16, 41.01); values treated=1,236
Outlier treatment completed without deleting rows.


## 4. Remove informational coupons

In [6]:
# Informational offers only provide information; they have no reward and no purchase
# difficulty. Remove them from both the offer catalog and the event table so the two outputs
# remain consistent. Transaction events are retained because they do not belong to a coupon.
informational_ids = set(
    portfolio.loc[portfolio['offer_type'].eq('informational'), 'id']
)
portfolio = portfolio.loc[~portfolio['id'].isin(informational_ids)].reset_index(drop=True)
events = events.loc[
    ~events['offer_id'].isin(informational_ids)
].reset_index(drop=True)

print(f'Removed informational offers: {len(informational_ids)}')
print(f'Remaining offers: {len(portfolio):,}')
print(f'Remaining events: {len(events):,}')
print('Remaining offer types:', portfolio['offer_type'].value_counts().to_dict())

Removed informational offers: 2
Remaining offers: 8
Remaining events: 280,071
Remaining offer types: {'bogo': 4, 'discount': 4}


## 5. Save the cleaned data

In [7]:
# Save JSON Lines because it preserves the original record-oriented format and can be
# streamed efficiently. Dates are written as ISO strings for portability.
def write_jsonl(frame, path):
    serializable = frame.copy()
    for column in serializable.columns:
        if pd.api.types.is_datetime64_any_dtype(serializable[column]):
            serializable[column] = serializable[column].dt.strftime('%Y-%m-%d')
    records = serializable.replace({np.nan: None}).to_dict(orient='records')
    with path.open('w', encoding='utf-8') as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + '\n')

clean_profile_path = PROCESSED_DIR / 'profile_cleaned.jsonl'
clean_portfolio_path = PROCESSED_DIR / 'portfolio_cleaned.jsonl'
clean_events_path = PROCESSED_DIR / 'transcript_cleaned.jsonl'
summary_path = PROCESSED_DIR / 'cleaning_summary.csv'

write_jsonl(profile, clean_profile_path)
write_jsonl(portfolio, clean_portfolio_path)
write_jsonl(events, clean_events_path)

summary = pd.DataFrame({
    'dataset': ['profile', 'portfolio', 'events'],
    'rows_saved': [len(profile), len(portfolio), len(events)],
    'columns_saved': [len(profile.columns), len(portfolio.columns), len(events.columns)]
})
summary.to_csv(summary_path, index=False)

print('Saved cleaned files:')
for path in [clean_profile_path, clean_portfolio_path, clean_events_path, summary_path]:
    print(f' - {path}')

Saved cleaned files:
 - D:\SanJose\sem2\CMPE252\data\processed\profile_cleaned.jsonl
 - D:\SanJose\sem2\CMPE252\data\processed\portfolio_cleaned.jsonl
 - D:\SanJose\sem2\CMPE252\data\processed\transcript_cleaned.jsonl
 - D:\SanJose\sem2\CMPE252\data\processed\cleaning_summary.csv
